In [1]:
import sys
sys.path.append("/data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR")

# Experimentation with Sklearn Models

This notebook evaluates Sklearn models on two UCI datasets that are more suitable for anomaly detection: `shuttle` and `arrhythmia`. The goal is to load the datasets from the RADAR static dataset module, reframe them as anomaly-detection benchmarks, and compare several Sklearn models using label-based and score-based metrics.

## Import Required Libraries

Import the necessary libraries for data loading, preprocessing, model training, and visualization.

In [7]:
# Import Required Libraries
import importlib

import numpy as np
import pandas as pd

from sklearn.covariance import EllipticEnvelope
from sklearn.linear_model import SGDOneClassSVM

from RADAR.static_data.algorithms import sklearn
import RADAR.metrics_module as metrics_module

metrics_module = importlib.reload(metrics_module)

## UCI Experiment: Shuttle and Arrhythmia

This benchmark uses two UCI datasets that are more appropriate for anomaly detection than the earlier classification-style datasets.

The anomaly-benchmark preparation logic now lives in `RADAR.static_data.anomaly_dataset_utils`, so the notebook only configures which datasets and labels to use.

In [8]:
import RADAR.static_data.anomaly_dataset_utils as anomaly_dataset_utils

anomaly_dataset_utils = importlib.reload(anomaly_dataset_utils)

uci_dataset_configs = {
    "shuttle": anomaly_dataset_utils.build_loaded_uci_anomaly_dataset(
        dataset_name="shuttle",
        normal_label=1,
        target_test_contamination=0.1,
        max_train_normals=8000,
        max_test_size=5000,
    ),
    "arrhythmia": anomaly_dataset_utils.build_loaded_uci_anomaly_dataset(
        dataset_name="arrhythmia",
        normal_label=1,
        target_test_contamination=0.1,
    ),
}

In [9]:
uci_summary_rows = []
for dataset_name, config in uci_dataset_configs.items():
    uci_summary_rows.append(
        {
            "dataset": dataset_name,
            "samples": config["n_samples"],
            "features": config["n_features"],
            "original_anomaly_ratio": round(config["original_positive_ratio"], 4),
            "benchmark_test_contamination": round(
                config["benchmark_test_positive_ratio"], 4
            ),
            "train_normals_used": config["train_normals"],
            "test_normals": config["test_normals"],
            "test_anomalies": config["test_anomalies"],
        }
    )

uci_summary_df = pd.DataFrame(uci_summary_rows)
display(uci_summary_df)

,dataset,samples,features,original_anomaly_ratio,benchmark_test_contamination,train_normals_used,test_normals,test_anomalies
0,shuttle,58000,7,0.214,0.1036,8000,4482,518
1,arrhythmia,452,279,0.458,0.0926,196,49,5


### Sklearn Anomaly Detection Experiment on Shuttle and Arrhythmia

This comparison uses the anomaly-detection models actually implemented in `RADAR.static_data.algorithms.sklearn`: `elliptic` and `sgdocsvm`.

`EllipticEnvelope` is only evaluated when the training set has more samples than features. For high-dimensional datasets such as `arrhythmia`, the notebook skips that model to avoid invalid covariance estimation.

In [14]:
uci_sklearn_models = [
    {
        "algorithm_": "elliptic",
        "model_kwargs": {"contamination": 0.1, "random_state": 42},
    },
    {
        "algorithm_": "sgdocsvm",
        "model_kwargs": {
            "nu": 0.1,
            "shuffle": True,
            "fit_intercept": True,
            "random_state": 42,
            "tol": 1e-6,
        },
    },
]

uci_results = []

for dataset_name, config in uci_dataset_configs.items():
    print(f"\nDataset: {dataset_name}")
    print(
        f"Training with normal-only samples: {config['train_normals']} | "
        f"Benchmark contamination: {config['benchmark_test_positive_ratio']:.3f}"
    )

    train_samples, train_features = config["X_train"].shape

    for model_params in uci_sklearn_models:
        algorithm_name = model_params["algorithm_"]
        model_kwargs = dict(model_params["model_kwargs"])

        if algorithm_name == "elliptic":
            model_kwargs["contamination"] = config["benchmark_test_positive_ratio"]
            if train_samples <= train_features:
                skip_reason = (
                    "skipped because EllipticEnvelope requires more training samples "
                    f"than features ({train_samples} <= {train_features})"
                )
                print(f"\nModel: {algorithm_name} | {skip_reason}")
                uci_results.append(
                    {
                        "dataset": dataset_name,
                        "algorithm": algorithm_name,
                        "status": "skipped",
                        "error": skip_reason,
                        "accuracy": np.nan,
                        "precision": np.nan,
                        "recall": np.nan,
                        "f1": np.nan,
                        "roc_auc_scores": np.nan,
                        "pr_auc_scores": np.nan,
                    }
                )
                continue

        model = sklearn.SkLearnAnomalyDetection(
            algorithm_=algorithm_name,
            **model_kwargs,
        )
        model.fit(config["X_train"])

        raw_predictions = model.predict(config["X_test"])
        scores = model.decision_function(config["X_test"])

        if raw_predictions is None or scores is None:
            failure_reason = "skipped due to fitting/scoring failure"
            print(f"\nModel: {algorithm_name} | {failure_reason}")
            uci_results.append(
                {
                    "dataset": dataset_name,
                    "algorithm": algorithm_name,
                    "status": "failed",
                    "error": failure_reason,
                    "accuracy": np.nan,
                    "precision": np.nan,
                    "recall": np.nan,
                    "f1": np.nan,
                    "roc_auc_scores": np.nan,
                    "pr_auc_scores": np.nan,
                }
            )
            continue

        raw_predictions = np.asarray(raw_predictions).ravel()
        predictions = (raw_predictions == -1).astype(int)
        scores = np.asarray(scores, dtype=float).ravel()

        accuracy = metrics_module.metric_accuracy(config["y_test"], predictions) / 100
        precision = metrics_module.metric_precision(config["y_test"], predictions)
        recall = metrics_module.metric_recall(config["y_test"], predictions)
        f1 = metrics_module.metric_F1score(config["y_test"], predictions)

        finite_scores = np.isfinite(scores)
        if finite_scores.all():
            roc_auc = metrics_module.metric_AUC_ROC_scores(config["y_test"], scores)
            pr_auc = metrics_module.metric_PR_AUC(config["y_test"], scores)
            score_note = ""
        else:
            roc_auc = np.nan
            pr_auc = np.nan
            score_note = " | score metrics skipped (NaN decision scores)"

        print(f"\nModel: {algorithm_name}{score_note}")
        metrics_module.print_metrics(
            ["Accuracy", "Precision", "Recall", "F1"],
            config["y_test"],
            predictions,
        )
        if finite_scores.all():
            print(f"ROC AUC (scores): {roc_auc:.3f}")
            print(f"PR AUC (scores): {pr_auc:.3f}")

        uci_results.append(
            {
                "dataset": dataset_name,
                "algorithm": algorithm_name,
                "status": "ok",
                "error": "",
                "accuracy": round(accuracy, 4),
                "precision": round(precision, 4),
                "recall": round(recall, 4),
                "f1": round(f1, 4),
                "roc_auc_scores": round(float(roc_auc), 4) if np.isfinite(roc_auc) else np.nan,
                "pr_auc_scores": round(float(pr_auc), 4) if np.isfinite(pr_auc) else np.nan,
            }
        )

uci_results_df = pd.DataFrame(uci_results).sort_values(
    ["dataset", "status", "pr_auc_scores", "roc_auc_scores"],
    ascending=[True, True, False, False],
    na_position="last",
).reset_index(drop=True)

display(uci_results_df)


Dataset: shuttle
Training with normal-only samples: 8000 | Benchmark contamination: 0.104

Model: elliptic
Accuracy: 87.640%
Precision: 0.434
Recall: 0.635
F1 Score: 0.516
ROC AUC (scores): 0.105
PR AUC (scores): 0.056

Model: sgdocsvm
Accuracy: 83.180%
Precision: 0.021
Recall: 0.014
F1 Score: 0.016
ROC AUC (scores): 0.838
PR AUC (scores): 0.462

Dataset: arrhythmia
Training with normal-only samples: 196 | Benchmark contamination: 0.093

Model: elliptic | skipped because EllipticEnvelope requires more training samples than features (196 <= 279)

Model: sgdocsvm
Accuracy: 90.741%
Precision: 0.000
Recall: 0.000
F1 Score: 0.000
ROC AUC (scores): 0.796
PR AUC (scores): 0.279


,dataset,algorithm,status,error,accuracy,precision,recall,f1,roc_auc_scores,pr_auc_scores
0,arrhythmia,sgdocsvm,ok,,0.9074,0.0000,0.0000,0.0000,0.7959,0.2794
1,arrhythmia,elliptic,skipped,skipped because EllipticEnvelope requires more...,NaN,NaN,NaN,NaN,NaN,NaN
2,shuttle,sgdocsvm,ok,,0.8318,0.0208,0.0135,0.0164,0.8380,0.4615
3,shuttle,elliptic,ok,,0.8764,0.4340,0.6351,0.5157,0.1045,0.0562


### Timing Comparison: RADAR vs Direct sklearn

### How to Interpret the Timing Table

- `dataset`: dataset on which the comparison was run (`shuttle` or `arrhythmia`).
- `algorithm`: sklearn anomaly model being evaluated.
- `average_platform_time_s`: average fit time in seconds using the RADAR wrapper.
- `average_base_time_s`: average fit time in seconds using the direct sklearn class.
- `speedup_base_over_platform`: ratio `average_base_time_s / average_platform_time_s`.
- `platform_roc_auc_scores`: score-based ROC-AUC obtained with the RADAR implementation.
- `base_roc_auc_scores`: score-based ROC-AUC obtained with the direct sklearn implementation.
- `roc_auc_diff`: difference `platform_roc_auc_scores - base_roc_auc_scores`.

### When Is One Better Than the Other?

- For **runtime**, RADAR is faster when `speedup_base_over_platform > 1`.
- If `speedup_base_over_platform < 1`, the direct sklearn implementation is faster.
- For **quality**, higher ROC-AUC is better.
- The ideal case is `speedup_base_over_platform > 1` and `roc_auc_diff >= 0`.

In [15]:
import time
from statistics import mean

from tqdm import tqdm

direct_sklearn_algorithms = {
    "elliptic": EllipticEnvelope,
    "sgdocsvm": SGDOneClassSVM,
}

uci_timing_results = []
num_repetitions = 30

for dataset_name, config in uci_dataset_configs.items():
    train_samples, train_features = config["X_train"].shape

    for model_params in uci_sklearn_models:
        algorithm_name = model_params["algorithm_"]
        shared_kwargs = dict(model_params["model_kwargs"])

        if algorithm_name == "elliptic":
            shared_kwargs["contamination"] = config["benchmark_test_positive_ratio"]
            if train_samples <= train_features:
                skip_reason = (
                    "skipped because EllipticEnvelope requires more training samples "
                    f"than features ({train_samples} <= {train_features})"
                )
                uci_timing_results.append(
                    {
                        "dataset": dataset_name,
                        "algorithm": algorithm_name,
                        "status": "skipped",
                        "error": skip_reason,
                        "average_platform_time_s": np.nan,
                        "average_base_time_s": np.nan,
                        "overhead_s": np.nan,
                        "speedup_base_over_platform": np.nan,
                        "platform_accuracy": np.nan,
                        "base_accuracy": np.nan,
                        "platform_roc_auc_scores": np.nan,
                        "base_roc_auc_scores": np.nan,
                        "roc_auc_diff": np.nan,
                    }
                )
                continue

        platform_execution_times = []
        platform_model = None
        platform_error = ""
        for _ in tqdm(
            range(num_repetitions),
            desc=f"Platform Execution Timing ({algorithm_name})",
        ):
            platform_model = sklearn.SkLearnAnomalyDetection(
                algorithm_=algorithm_name,
                **shared_kwargs,
            )
            start_time = time.time()
            platform_model.fit(config["X_train"])
            platform_execution_times.append(time.time() - start_time)
            test_predictions = platform_model.predict(config["X_test"])
            test_scores = platform_model.decision_function(config["X_test"])
            if test_predictions is None or test_scores is None:
                platform_error = "fit/predict failure"
                break

        direct_model_cls = direct_sklearn_algorithms[algorithm_name]
        direct_execution_times = []
        direct_model = None
        direct_error = ""
        for _ in tqdm(
            range(num_repetitions),
            desc=f"Direct Execution Timing ({algorithm_name})",
        ):
            direct_model = direct_model_cls(**shared_kwargs)
            start_time = time.time()
            try:
                direct_model.fit(config["X_train"])
                direct_execution_times.append(time.time() - start_time)
                direct_predictions_check = direct_model.predict(config["X_test"])
                direct_scores_check = direct_model.decision_function(config["X_test"])
                if direct_predictions_check is None or direct_scores_check is None:
                    direct_error = "fit/predict failure"
                    break
            except Exception as exc:
                direct_error = str(exc)
                direct_execution_times.append(time.time() - start_time)
                break

        average_platform_time = mean(platform_execution_times) if platform_execution_times else np.nan
        average_direct_time = mean(direct_execution_times) if direct_execution_times else np.nan

        if platform_error or direct_error:
            uci_timing_results.append(
                {
                    "dataset": dataset_name,
                    "algorithm": algorithm_name,
                    "status": "failed",
                    "error": platform_error or direct_error,
                    "average_platform_time_s": round(average_platform_time, 4)
                    if np.isfinite(average_platform_time)
                    else np.nan,
                    "average_base_time_s": round(average_direct_time, 4)
                    if np.isfinite(average_direct_time)
                    else np.nan,
                    "overhead_s": np.nan,
                    "speedup_base_over_platform": np.nan,
                    "platform_accuracy": np.nan,
                    "base_accuracy": np.nan,
                    "platform_roc_auc_scores": np.nan,
                    "base_roc_auc_scores": np.nan,
                    "roc_auc_diff": np.nan,
                }
            )
            continue

        platform_raw_predictions = np.asarray(
            platform_model.predict(config["X_test"])
        ).ravel()
        platform_predictions = (platform_raw_predictions == -1).astype(int)
        platform_scores = np.asarray(
            platform_model.decision_function(config["X_test"]),
            dtype=float,
        ).ravel()
        platform_roc_auc = (
            metrics_module.metric_AUC_ROC_scores(config["y_test"], platform_scores)
            if np.isfinite(platform_scores).all()
            else np.nan
        )

        direct_raw_predictions = np.asarray(direct_model.predict(config["X_test"])).ravel()
        direct_predictions = (direct_raw_predictions == -1).astype(int)
        direct_scores = np.asarray(
            direct_model.decision_function(config["X_test"]),
            dtype=float,
        ).ravel()
        direct_roc_auc = (
            metrics_module.metric_AUC_ROC_scores(config["y_test"], direct_scores)
            if np.isfinite(direct_scores).all()
            else np.nan
        )

        overhead = average_platform_time - average_direct_time

        uci_timing_results.append(
            {
                "dataset": dataset_name,
                "algorithm": algorithm_name,
                "status": "ok",
                "error": "",
                "average_platform_time_s": round(average_platform_time, 4),
                "average_base_time_s": round(average_direct_time, 4),
                "overhead_s": round(overhead, 4),
                "speedup_base_over_platform": round(
                    average_direct_time / average_platform_time, 4
                ) if average_platform_time > 0 else np.nan,
                "platform_accuracy": round(
                    metrics_module.metric_accuracy(config["y_test"], platform_predictions) / 100,
                    4,
                ),
                "base_accuracy": round(
                    metrics_module.metric_accuracy(config["y_test"], direct_predictions) / 100,
                    4,
                ),
                "platform_roc_auc_scores": round(float(platform_roc_auc), 4)
                if np.isfinite(platform_roc_auc)
                else np.nan,
                "base_roc_auc_scores": round(float(direct_roc_auc), 4)
                if np.isfinite(direct_roc_auc)
                else np.nan,
                "roc_auc_diff": round(float(platform_roc_auc - direct_roc_auc), 4)
                if np.isfinite(platform_roc_auc) and np.isfinite(direct_roc_auc)
                else np.nan,
            }
        )

uci_timing_df = pd.DataFrame(uci_timing_results).sort_values(
    ["dataset", "status", "speedup_base_over_platform"],
    ascending=[True, True, False],
    na_position="last",
).reset_index(drop=True)

display(uci_timing_df)

Direct Execution Timing (sgdocsvm): 100%|██████████| 30/30 [00:00<00:00, 906.81it/s]


,dataset,algorithm,status,error,average_platform_time_s,average_base_time_s,overhead_s,speedup_base_over_platform,platform_accuracy,base_accuracy,platform_roc_auc_scores,base_roc_auc_scores,roc_auc_diff
0,arrhythmia,sgdocsvm,ok,,0.0009,0.0009,0.0000,0.9757,0.9074,0.9074,0.7959,0.7959,0.0
1,arrhythmia,elliptic,skipped,skipped because EllipticEnvelope requires more...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,shuttle,sgdocsvm,ok,,0.0034,0.0034,0.0001,0.9845,0.8318,0.8318,0.8380,0.8380,0.0
3,shuttle,elliptic,ok,,0.8399,0.8017,0.0382,0.9545,0.8764,0.8764,0.1045,0.1045,0.0


In [13]:
from pathlib import Path

results_dir = Path("/data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/results")
results_dir.mkdir(parents=True, exist_ok=True)

results_main_path = results_dir / "uci_sklearn_results.csv"
results_timing_path = results_dir / "uci_sklearn_timing_results.csv"

uci_results_df.to_csv(results_main_path, index=False)
uci_timing_df.to_csv(results_timing_path, index=False)

print(f"Saved main results to: {results_main_path}")
print(f"Saved timing results to: {results_timing_path}")

Saved main results to: /data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/results/uci_sklearn_results.csv
Saved timing results to: /data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/results/uci_sklearn_timing_results.csv
